In [ ]:
# Section 0: Import and Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from scipy.optimize import minimize_scalar
import ipywidgets as widgets
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ All libraries imported successfully")

In [ ]:
# Greeks Calculation Functions
def black_scholes_greeks(S, K, T, sigma, r=0.06, q=0.02, option_type='call'):
    """
    Compute Black-Scholes Greeks for European options.
    
    Parameters:
    S: spot price
    K: strike price
    T: time to expiry (years)
    sigma: volatility (annualized)
    r: risk-free rate
    q: dividend yield
    option_type: 'call' or 'put'
    
    Returns: dict with delta, gamma, vega, theta, rho
    """
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    # Greeks (same for call and put, but sign differs on some)
    delta_call = np.exp(-q * T) * norm.cdf(d1)
    delta_put = -np.exp(-q * T) * norm.cdf(-d1)
    
    gamma = np.exp(-q * T) * norm.pdf(d1) / (S * sigma * np.sqrt(T))
    
    vega = S * np.exp(-q * T) * norm.pdf(d1) * np.sqrt(T) / 100  # per 1% vol change
    
    theta_call = (-S * np.exp(-q * T) * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) 
                  - r * K * np.exp(-r * T) * norm.cdf(d2) 
                  + q * S * np.exp(-q * T) * norm.cdf(d1)) / 365
    
    theta_put = (-S * np.exp(-q * T) * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) 
                 + r * K * np.exp(-r * T) * norm.cdf(-d2) 
                 - q * S * np.exp(-q * T) * norm.cdf(-d1)) / 365
    
    rho_call = K * T * np.exp(-r * T) * norm.cdf(d2) / 100
    rho_put = -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100
    
    if option_type == 'call':
        return {'Delta': delta_call, 'Gamma': gamma, 'Vega': vega, 'Theta': theta_call, 'Rho': rho_call}
    else:
        return {'Delta': delta_put, 'Gamma': gamma, 'Vega': vega, 'Theta': theta_put, 'Rho': rho_put}

# Interactive Greeks Explorer
print("INTERACTIVE GREEKS CALCULATOR\n" + "="*50)

spot_slider = widgets.FloatSlider(value=21000, min=18000, max=24000, step=100, description='Spot:', layout=widgets.Layout(width='400px'))
strike_slider = widgets.FloatSlider(value=21000, min=18000, max=24000, step=100, description='Strike:', layout=widgets.Layout(width='400px'))
vol_slider = widgets.FloatSlider(value=0.18, min=0.05, max=0.50, step=0.01, description='Vol (σ):', layout=widgets.Layout(width='400px'))
dte_slider = widgets.FloatSlider(value=5, min=1, max=60, step=1, description='DTE:', layout=widgets.Layout(width='400px'))
option_radio = widgets.RadioButtons(options=['CALL', 'PUT'], description='Type:', layout=widgets.Layout(width='200px'))

def update_greeks(spot, strike, vol, dte, option_type):
    T = dte / 365
    greeks = black_scholes_greeks(spot, strike, T, vol, option_type=option_type.lower())
    
    print(f"\n{'─'*60}")
    print(f"SPOT: {spot:,.0f} | STRIKE: {strike:,.0f} | {option_type} | DTE: {dte:.0f}")
    print(f"VOLATILITY: {vol*100:.1f}% | TIME TO EXPIRY: {T*365:.0f} days")
    print(f"{'─'*60}")
    
    for greek, value in greeks.items():
        if greek == 'Delta':
            print(f"{greek:8s}: {value:8.4f}  (price moves {value*100:.2f}% per 1% spot move)")
        elif greek == 'Gamma':
            print(f"{greek:8s}: {value:8.6f}  (delta changes by {value*100:.4f}% per 1% spot move)")
        elif greek == 'Vega':
            print(f"{greek:8s}: {value:8.4f}  (price changes by {value:.4f} per 1% vol change)")
        elif greek == 'Theta':
            print(f"{greek:8s}: {value:8.4f}  (daily decay; ${value*100:.2f} per day)")
        elif greek == 'Rho':
            print(f"{greek:8s}: {value:8.4f}  (price changes by {value:.4f} per 1% rate change)")

display(widgets.HBox([widgets.VBox([spot_slider, strike_slider, vol_slider, dte_slider, option_radio])]));
widgets.interact(update_greeks, spot=spot_slider, strike=strike_slider, vol=vol_slider, dte=dte_slider, option_type=option_radio);

In [ ]:
# Regime Classification Engine
print("\n" + "="*70)
print("REGIME CLASSIFICATION SYSTEM")
print("="*70)

# Regime policy thresholds (from codebase)
REGIME_POLICY = {
    'POSITIVE_GAMMA': {'composite_delta': -3, 'strength_delta': -2, 'position_mult': 1.2, 'confidence_mult': 1.1},
    'NEGATIVE_GAMMA': {'composite_delta': +5, 'strength_delta': +3, 'position_mult': 0.7, 'confidence_mult': 0.7},
    'NEUTRAL_GAMMA': {'composite_delta': 0, 'strength_delta': 0, 'position_mult': 1.0, 'confidence_mult': 1.0},
    'VOL_EXPANSION': {'composite_delta': +2, 'position_mult': 0.85, 'holding_delta': -30},
    'VOL_COMPRESSION': {'composite_delta': -1, 'position_mult': 1.1, 'holding_delta': +30},
    'RISK_OFF': {'composite_delta': +5, 'position_mult': 0.65, 'prob_requirement': 0.70}
}

def classify_regime(gex_ratio, vol_slope, vix_level, rbi_rate_change, usd_inr_move):
    """
    Classify market regimes based on indicator inputs.
    
    gex_ratio: gamma skew (positive = dealers short puts, long calls)
    vol_slope: vol term structure slope (positive = contango)
    vix_level: VIX index level
    rbi_rate_change: RBI rate shift (bps)
    usd_inr_move: USD/INR move (%)
    """
    results = {}
    
    # Gamma regime
    if gex_ratio > 0.05:
        gamma_regime = 'POSITIVE_GAMMA'
        gamma_desc = 'Dealers SHORT puts/LONG calls → Vol support, upside bias'
    elif gex_ratio < -0.05:
        gamma_regime = 'NEGATIVE_GAMMA'
        gamma_desc = 'Dealers LONG puts/SHORT calls → Vol vulnerability, downside bias'
    else:
        gamma_regime = 'NEUTRAL_GAMMA'
        gamma_desc = 'Balanced dealer positioning'
    
    # Vol regime
    if vol_slope > 0.02:
        vol_regime = 'VOL_EXPANSION'
        vol_desc = 'Vol curve in contango → Risk-off signal; reduce sizing'
    elif vol_slope < -0.02:
        vol_regime = 'VOL_COMPRESSION'
        vol_desc = 'Vol curve in backwardation → Risk-on signal; increase sizing'
    else:
        vol_regime = 'VOL_NEUTRAL'
        vol_desc = 'Vol curve flat'
    
    # Global risk
    if vix_level > 25 or usd_inr_move > 0.5 or rbi_rate_change < -25:
        risk_state = 'HIGH_RISK'
        risk_desc = 'Stress conditions; flight to safety'
    elif vix_level < 12 and usd_inr_move < -0.2 and rbi_rate_change > 0:
        risk_state = 'LOW_RISK'
        risk_desc = 'Benign conditions; risk appetite'
    else:
        risk_state = 'NORMAL_RISK'
        risk_desc = 'Baseline market state'
    
    return {
        'gamma': gamma_regime, 'gamma_desc': gamma_desc,
        'vol': vol_regime, 'vol_desc': vol_desc,
        'risk': risk_state, 'risk_desc': risk_desc
    }

def apply_policy_multipliers(base_score, regime_dict):
    """Apply policy multipliers based on regimes."""
    multiplier = 1.0
    
    # Apply gamma multiplier
    gamma_policy = REGIME_POLICY.get(regime_dict['gamma'], {})
    multiplier *= gamma_policy.get('position_mult', 1.0)
    
    # Apply vol multiplier
    vol_policy = REGIME_POLICY.get(regime_dict['vol'], {})
    multiplier *= vol_policy.get('position_mult', 1.0)
    
    return base_score * multiplier

# Interactive Regime Classifier
print("\nINTERACTIVE REGIME CLASSIFIER\n" + "─"*70)

gex_slider = widgets.FloatSlider(value=0.02, min=-0.15, max=0.15, step=0.01, description='GEX Ratio:', layout=widgets.Layout(width='400px'))
vol_slope_slider = widgets.FloatSlider(value=0.005, min=-0.05, max=0.05, step=0.005, description='Vol Slope:', layout=widgets.Layout(width='400px'))
vix_slider = widgets.FloatSlider(value=16, min=8, max=40, step=1, description='VIX Level:', layout=widgets.Layout(width='400px'))
rbi_slider = widgets.FloatSlider(value=0, min=-100, max=100, step=10, description='RBI Δ (bps):', layout=widgets.Layout(width='400px'))
usd_slider = widgets.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.1, description='USD/INR Δ (%):', layout=widgets.Layout(width='400px'))

def update_regime(gex, vol_slope, vix, rbi, usd):
    regime = classify_regime(gex, vol_slope, vix, rbi, usd)
    
    print(f"\n{'REGIME CLASSIFICATION':^70}")
    print(f"{'─'*70}")
    print(f"\n1. GAMMA REGIME:         {regime['gamma']}")
    print(f"   → {regime['gamma_desc']}")
    print(f"\n2. VOLATILITY REGIME:    {regime['vol']}")
    print(f"   → {regime['vol_desc']}")
    print(f"\n3. GLOBAL RISK STATE:    {regime['risk']}")
    print(f"   → {regime['risk_desc']}")
    
    # Show policy multipliers
    gamma_mult = REGIME_POLICY[regime['gamma']].get('position_mult', 1.0)
    vol_mult = REGIME_POLICY[regime['vol']].get('position_mult', 1.0)
    total_mult = gamma_mult * vol_mult
    
    print(f"\n{'APPLIED MULTIPLIERS':^70}")
    print(f"{'─'*70}")
    print(f"Gamma Multiplier:     {gamma_mult:.2f}x")
    print(f"Vol Multiplier:       {vol_mult:.2f}x")
    print(f"COMBINED:             {total_mult:.2f}x")
    print(f"\n→ A 2-lot signal becomes {2*total_mult:.2f} lots under this regime")

display(widgets.HBox([widgets.VBox([gex_slider, vol_slope_slider, vix_slider, rbi_slider, usd_slider])]));
widgets.interact(update_regime, gex=gex_slider, vol_slope=vol_slope_slider, vix=vix_slider, rbi=rbi_slider, usd=usd_slider);

In [ ]:
# Probability Calibration (Platt Scaling)
print("\n" + "="*70)
print("PROBABILITY CALIBRATION: PLATT SCALING")
print("="*70)

def sigmoid(x):
    """Sigmoid function for Platt scaling."""
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def platt_transform(score, A, B):
    """Apply Platt scaling to raw score."""
    return sigmoid(A * score + B)

# Example calibration curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Calibration curves with different A, B values
scores = np.linspace(0, 100, 200)

ax1.plot(scores, sigmoid(0.03 * scores - 2.0), 'b-', linewidth=2.5, label='Conservative (A=0.03, B=-2.0)')
ax1.plot(scores, sigmoid(0.05 * scores - 2.5), 'g-', linewidth=2.5, label='Moderate (A=0.05, B=-2.5)')
ax1.plot(scores, sigmoid(0.08 * scores - 4.0), 'r-', linewidth=2.5, label='Aggressive (A=0.08, B=-4.0)')
ax1.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax1.axvline(x=50, color='gray', linestyle='--', alpha=0.5)
ax1.set_xlabel('Raw Model Score (0-100)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Calibrated Probability', fontsize=11, fontweight='bold')
ax1.set_title('Platt Scaling: Score → Probability', fontsize=12, fontweight='bold')
ax1.set_ylim([0, 1])
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Right plot: Impact of A and B parameters
ax2.text(0.5, 0.95, 'Platt Scaling Impact', ha='center', va='top', fontsize=12, fontweight='bold', transform=ax2.transAxes)

calibration_text = """
Raw Score → Calibrated Probability

A = Slope (sensitivity):
  • High A: Steep curve (more confident)
  • Low A: Gradual curve (less confident)

B = Intercept (bias shift):
  • High B: Shift curve right (boost prob)
  • Low B: Shift curve left (reduce prob)

Typical Values:
  • Conservative: A=0.03, B=-2.0
  • Moderate:     A=0.05, B=-2.5
  • Aggressive:   A=0.08, B=-4.0

Example:
  Score=60 with Conservative (A=0.03, B=-2.0)
  P = sigmoid(0.03×60 - 2.0) = 0.346 = 34.6%

  Same score with Aggressive (A=0.08, B=-4.0)
  P = sigmoid(0.08×60 - 4.0) = 0.182 = 18.2%
"""

ax2.text(0.05, 0.85, calibration_text, ha='left', va='top', fontsize=9, 
         family='monospace', transform=ax2.transAxes, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
ax2.axis('off')

plt.tight_layout()
plt.show()

print("\n✓ Calibration curves displayed above")
print("\nKey Insight: Same raw score maps to DIFFERENT probabilities under")
print("different calibration regimes. The right A, B values depend on your")
print("training data and your risk tolerance.")

In [ ]:
# Overnight Risk & Gap Mechanics
print("\n" + "="*70)
print("OVERNIGHT RISK & GAP MECHANICS")
print("="*70)

def gap_probability(vix_change, has_earnings, sp500_overnight_move):
    """
    Compute probability of gap > 2% for next open.
    
    From trading guide Chapter 22:
    P(gap > 2%) ≈ 0.20 + 0.30 × (ΔVIx/20) + 0.50 × I_earnings
    """
    base_prob = 0.20
    vix_term = 0.30 * (vix_change / 20)
    earnings_term = 0.50 if has_earnings else 0.0
    
    total_prob = base_prob + vix_term + earnings_term
    return np.clip(total_prob, 0, 1)

def overnight_hold_decision(global_risk, gamma_regime, spot_vs_flip, upcoming_events):
    """
    Engine overnight hold decision.
    
    TRUE if: RISK_STATE not HIGH/EXTREME AND gamma not NEGATIVE AND no major events
    FALSE otherwise
    """
    if global_risk in ['HIGH_RISK', 'EXTREME_RISK']:
        return False, 'HIGH_RISK state → Exit before close'
    
    if gamma_regime == 'NEGATIVE_GAMMA' and spot_vs_flip == 'BELOW_FLIP':
        return False, 'NEGATIVE_GAMMA + BELOW_FLIP → Exit before close'
    
    if upcoming_events > 1:
        return False, f'{upcoming_events} upcoming events → Exit before close'
    
    return True, 'All checks pass → Safe to hold overnight'

# Interactive Gap Calculator
print("\nINTERACTIVE GAP PROBABILITY CALCULATOR\n" + "─"*70)

gap_vix_slider = widgets.FloatSlider(value=2.0, min=-5, max=10, step=0.5, description='ΔVIx:', layout=widgets.Layout(width='400px'))
gap_earnings_check = widgets.Checkbox(value=False, description='Earnings nearby (48h)', layout=widgets.Layout(width='400px'))
gap_sp500_slider = widgets.FloatSlider(value=0.0, min=-5, max=5, step=0.5, description='S&P 500 Δ (%):', layout=widgets.Layout(width='400px'))

def update_gap_prob(dvix, earnings, sp500_move):
    gap_prob = gap_probability(dvix, earnings, sp500_move)
    
    # Gap size estimation
    if sp500_move < -3:
        gap_size = 300  # 2.5-4% = 250-400 pts
        gap_desc = 'SEVERE: Panic liquidation expected'
    elif sp500_move < -2:
        gap_size = 200  # 1.5-2.5% = 100-200 pts
        gap_desc = 'LARGE: Significant overnight move'
    elif sp500_move < -1:
        gap_size = 100  # 1% ~ 50-100 pts
        gap_desc = 'MODERATE: Normal spillover'
    else:
        gap_size = 30  # Minor
        gap_desc = 'SMALL: Minor overnight drift'
    
    print(f"\n{'GAP PROBABILITY FORECAST':^70}")
    print(f"{'─'*70}")
    print(f"P(Gap > 2%)  : {gap_prob*100:6.2f}%")
    print(f"Expected gap : ~{gap_size:3.0f} points  ({gap_desc})")
    print(f"\nBreakdown:")
    print(f"  Base prob        : 20.0%")
    print(f"  VIX contribution : {0.30 * (dvix/20)*100:5.2f}%")
    print(f"  Earnings bonus   : {'50.0%' if earnings else ' 0.0%'}")
    print(f"  {'─'*60}")
    print(f"  TOTAL            : {gap_prob*100:5.2f}%")
    
    if gap_prob > 0.40:
        recommendation = "🔴 HIGH gap risk → Exit before close or reduce size"
    elif gap_prob > 0.25:
        recommendation = "🟡 MODERATE gap risk → Consider exit or tight stop"
    else:
        recommendation = "🟢 LOW gap risk → Safe to hold overnight"
    
    print(f"\n{recommendation}")

display(widgets.HBox([widgets.VBox([gap_vix_slider, gap_earnings_check, gap_sp500_slider])]));
widgets.interact(update_gap_prob, dvix=gap_vix_slider, earnings=gap_earnings_check, sp500_move=gap_sp500_slider);

# Overnight Hold Decision Tree
print("\n" + "─"*70)
print("OVERNIGHT HOLD DECISION TREE")
print("─"*70)

decision_risk = widgets.RadioButtons(options=['NORMAL_RISK', 'HIGH_RISK', 'EXTREME_RISK'], description='Global Risk:')
decision_gamma = widgets.RadioButtons(options=['POSITIVE_GAMMA', 'NEUTRAL_GAMMA', 'NEGATIVE_GAMMA'], description='Gamma:')
decision_spot = widgets.RadioButtons(options=['ABOVE_FLIP', 'BELOW_FLIP'], description='Spot vs Flip:')
decision_events = widgets.FloatSlider(value=0, min=0, max=3, step=1, description='# Upcoming Events:')

def update_hold_decision(risk, gamma, spot_vs_flip, events):
    hold_ok, reason = overnight_hold_decision(risk, gamma, spot_vs_flip, int(events))
    
    print(f"\n{'OVERNIGHT HOLD DECISION':^70}")
    print(f"{'─'*70}")
    print(f"Global Risk:      {risk}")
    print(f"Gamma Regime:     {gamma}")
    print(f"Spot vs Flip:     {spot_vs_flip}")
    print(f"Events Pending:   {int(events)}")
    print(f"\n{'DECISION':^70}")
    
    if hold_ok:
        print(f"✅ HOLD OVERNIGHT")
    else:
        print(f"❌ EXIT BEFORE CLOSE (15:15 IST)")
    
    print(f"\nReason: {reason}")

display(widgets.HBox([widgets.VBox([decision_risk, decision_gamma, decision_spot, decision_events])]));
widgets.interact(update_hold_decision, risk=decision_risk, gamma=decision_gamma, spot_vs_flip=decision_spot, events=decision_events);

In [ ]:
# Cross-Market Signals & Risk Transmission
print("\n" + "="*70)
print("CROSS-MARKET SIGNAL ANALYSIS")
print("="*70)

def compute_cross_market_score(sp500_change, usd_inr_change, oil_change, vix_change):
    """
    Compute cross-market risk score from multiple assets.
    
    Weights (from engine):
    - Futures: 40% (lead indicator)
    - USD/INR: 25% (FII flow, corporate stress)
    - Oil: 20% (inflation gauge)
    - VIX: 15% (volatility contagion)
    """
    weights = {
        'futures': 0.40,
        'usd': 0.25,
        'oil': 0.20,
        'vix': 0.15
    }
    
    # Normalize individual signals to 0-100 scale
    futures_score = 50 - np.clip(sp500_change * 30, -50, 50)  # Inverse; down = higher risk
    usd_score = 50 + np.clip(usd_inr_change * 40, -50, 50)    # Strong USD = risk-off
    oil_score = 50 - np.clip(oil_change * 20, -50, 50)        # Oil crash = risk-off
    vix_score = 50 + np.clip(vix_change * 5, -50, 50)         # VIX spike = risk-off
    
    # Composite score (higher = more risk-off)
    composite = (weights['futures'] * futures_score + 
                weights['usd'] * usd_score + 
                weights['oil'] * oil_score + 
                weights['vix'] * vix_score)
    
    return {
        'composite': composite,
        'futures': futures_score,
        'usd': usd_score,
        'oil': oil_score,
        'vix': vix_score
    }

# Interactive Cross-Market Scorer
print("\nINTERACTIVE CROSS-MARKET RISK SCORER\n" + "─"*70)

cm_sp500_slider = widgets.FloatSlider(value=0.0, min=-5, max=2, step=0.5, description='S&P 500 Δ (%):', layout=widgets.Layout(width='400px'))
cm_usd_slider = widgets.FloatSlider(value=0.0, min=-1.0, max=2.0, step=0.2, description='USD/INR Δ (%):', layout=widgets.Layout(width='400px'))
cm_oil_slider = widgets.FloatSlider(value=0.0, min=-10, max=5, step=1.0, description='Crude Oil Δ (%):', layout=widgets.Layout(width='400px'))
cm_vix_slider = widgets.FloatSlider(value=0.0, min=-5, max=20, step=1.0, description='VIX Δ (pts):', layout=widgets.Layout(width='400px'))

def update_cross_market(sp500, usd, oil, vix):
    scores = compute_cross_market_score(sp500, usd, oil, vix)
    
    print(f"\n{'CROSS-MARKET RISK ANALYSIS':^70}")
    print(f"{'─'*70}")
    
    # Component breakdown
    print(f"\nComponent Scores (0=Risk-On, 100=Risk-Off):")
    print(f"  S&P 500 Overnight:   {scores['futures']:5.1f}  ({'📈' if sp500 > 0 else '📉'} {sp500:+.1f}%)")
    print(f"  USD/INR (Currency):  {scores['usd']:5.1f}  ({'💪' if usd > 0 else '😶'} {usd:+.2f}%)")
    print(f"  Crude Oil:           {scores['oil']:5.1f}  ({'📈' if oil > 0 else '📉'} {oil:+.1f}%)")
    print(f"  VIX Level:           {scores['vix']:5.1f}  ({'😰' if vix > 0 else '😌'} {vix:+.1f}pts)")
    
    print(f"\n{'COMPOSITE CROSS-MARKET SCORE':^70}")
    print(f"{'─'*70}")
    print(f"  Score: {scores['composite']:.1f}/100")
    
    # Interpretation
    if scores['composite'] > 65:
        signal = "🔴 SEVERE RISK-OFF"
        action = "EXIT signals / Reduce size / Tighten stops / Consider skipping"
    elif scores['composite'] > 55:
        signal = "🟠 MODERATE RISK-OFF"
        action = "DOWNGRADE confidence / Reduce size by 20-30% / Exit rallies"
    elif scores['composite'] < 40:
        signal = "🟢 MODERATE RISK-ON"
        action = "UPGRADE confidence / Increase size / More aggressive entry"
    else:
        signal = "🟡 NEUTRAL / BALANCED"
        action = "TRADE AS NORMAL / Baseline regime"
    
    print(f"\nSignal: {signal}")
    print(f"Action: {action}")
    
    # Transmission mechanism
    print(f"\n{'TRANSMISSION TO NIFTY':^70}")
    print(f"{'─'*70}")
    
    if sp500 < -2:
        print(f"S&P down {sp500:.1f}% → NIFTY gap down 100-200 pts expected")
    if usd > 0.5:
        print(f"USD/INR up {usd:.2f}% → FII sellers / corporates stressed → downside")
    if oil < -5:
        print(f"Oil crash {oil:.1f}% → Deflation signal → safe havens (gold/bonds) attract flows → downside")
    if vix > 5:
        print(f"VIX +{vix:.1f}pts → Volatility contagion → IV spikes in NIFTY puts")

display(widgets.HBox([widgets.VBox([cm_sp500_slider, cm_usd_slider, cm_oil_slider, cm_vix_slider])]));
widgets.interact(update_cross_market, sp500=cm_sp500_slider, usd=cm_usd_slider, oil=cm_oil_slider, vix=cm_vix_slider);

In [ ]:
# Options Flow Analysis & Dealer Positioning
print("\n" + "="*70)
print("OPTIONS FLOW ANALYSIS")
print("="*70)

def compute_flow_imbalance(call_volume, call_price, call_delta, 
                          put_volume, put_price, put_delta):
    """
    Compute delta-weighted options flow imbalance.
    
    From trading guide Chapter 19:
    Flow = call_notional / put_notional
    Where notional = Σ(volume × price × |delta|)
    """
    call_notional = call_volume * call_price * abs(call_delta)
    put_notional = put_volume * put_price * abs(put_delta)
    
    if put_notional == 0:
        ratio = float('inf') if call_notional > 0 else 0
    else:
        ratio = call_notional / put_notional
    
    # Classify flow
    if ratio >= 1.2:
        flow_signal = 'BULLISH_FLOW'
        signal_desc = 'More bullish buyers; upside bias'
    elif ratio <= 0.83:
        flow_signal = 'BEARISH_FLOW'
        signal_desc = 'More bearish buyers; downside bias'
    else:
        flow_signal = 'NEUTRAL_FLOW'
        signal_desc = 'Balanced call/put interest; no directional tilt'
    
    return {
        'call_notional': call_notional,
        'put_notional': put_notional,
        'ratio': ratio,
        'signal': flow_signal,
        'description': signal_desc
    }

# Interactive Flow Calculator
print("\nINTERACTIVE OPTIONS FLOW ANALYZER\n" + "─"*70)

call_vol_slider = widgets.FloatSlider(value=15000, min=0, max=50000, step=1000, description='Call Volume:', layout=widgets.Layout(width='400px'))
call_price_slider = widgets.FloatSlider(value=85, min=10, max=300, step=5, description='Call Price (₹):', layout=widgets.Layout(width='400px'))
call_delta_slider = widgets.FloatSlider(value=0.60, min=0, max=1.0, step=0.05, description='Call |Delta|:', layout=widgets.Layout(width='400px'))

put_vol_slider = widgets.FloatSlider(value=18000, min=0, max=50000, step=1000, description='Put Volume:', layout=widgets.Layout(width='400px'))
put_price_slider = widgets.FloatSlider(value=65, min=10, max=300, step=5, description='Put Price (₹):', layout=widgets.Layout(width='400px'))
put_delta_slider = widgets.FloatSlider(value=0.45, min=0, max=1.0, step=0.05, description='Put |Delta|:', layout=widgets.Layout(width='400px'))

def update_flow(c_vol, c_price, c_delta, p_vol, p_price, p_delta):
    flow = compute_flow_imbalance(c_vol, c_price, c_delta, p_vol, p_price, p_delta)
    
    print(f"\n{'OPTIONS FLOW ANALYSIS':^70}")
    print(f"{'─'*70}")
    
    print(f"\nATM SLICE INPUT:")
    print(f"  Calls:  {c_vol:>8,.0f} contracts × ₹{c_price:>6.0f} × Δ{c_delta:.2f}")
    print(f"  Puts:   {p_vol:>8,.0f} contracts × ₹{p_price:>6.0f} × Δ{p_delta:.2f}")
    
    print(f"\nDELTA-WEIGHTED NOTIONAL:")
    print(f"  Call Notional: ₹ {flow['call_notional']:>15,.2f}")
    print(f"  Put Notional:  ₹ {flow['put_notional']:>15,.2f}")
    
    print(f"\n{'FLOW SIGNAL':^70}")
    print(f"{'─'*70}")
    print(f"  Ratio (Call/Put):  {flow['ratio']:.3f}")
    print(f"  Signal:            {flow['signal']}")
    print(f"  Description:       {flow['description']}")
    
    # Visual distribution
    total_notional = flow['call_notional'] + flow['put_notional']
    call_pct = (flow['call_notional'] / total_notional * 100) if total_notional > 0 else 50
    put_pct = 100 - call_pct
    
    print(f"\n  Call Flow: {call_pct:.1f}% {'█' * int(call_pct/5)} {int(call_pct)}%")
    print(f"  Put Flow:  {put_pct:.1f}% {'█' * int(put_pct/5)} {int(put_pct)}%")
    
    # Trading implication
    print(f"\n{'TRADING IMPLICATION':^70}")
    print(f"{'─'*70}")
    
    if flow['signal'] == 'BULLISH_FLOW':
        print("✅ FAVOR LONG CALLS / SELL PUTS")
        print("   Reasoning: Flow suggests bullish conviction")
        print("   Risk: Reversal if dealers hedged the put buyers")
    elif flow['signal'] == 'BEARISH_FLOW':
        print("✅ FAVOR LONG PUTS / SELL CALLS")
        print("   Reasoning: Flow suggests bearish conviction")
        print("   Risk: Reversal if dealers hedged the call buyers")
    else:
        print("⚠️  NO DIRECTIONAL TILT FROM FLOW")
        print("   Recommendation: Rely on other signals (gamma, regime, momentum)")

display(widgets.HBox([widgets.VBox([
    widgets.VBox([widgets.HTML("<b>CALL SIDE:</b>"), call_vol_slider, call_price_slider, call_delta_slider]),
    widgets.VBox([widgets.HTML("<b>PUT SIDE:</b>"), put_vol_slider, put_price_slider, put_delta_slider])
])]));

widgets.interact(update_flow, 
                c_vol=call_vol_slider, c_price=call_price_slider, c_delta=call_delta_slider,
                p_vol=put_vol_slider, p_price=put_price_slider, p_delta=put_delta_slider);

---

## Summary: Quick Reference

This notebook covered the **five most important concepts** for using the Options Quant Engine:

| Concept | Use Case | Key Formula |
|---------|----------|-------------|
| **Greeks** (Δ, Γ, Ν, Θ, Ρ) | Position risk measurement | $\Delta = \frac{\partial C}{\partial S}$ |
| **Regimes** (Gamma/Vol/Risk) | Context for signal weighting | $\text{Size Multiplier} = M_\gamma \times M_{vol}$ |
| **Probability Calibration** | Score → True probability | $P = \sigmoid(A \cdot \text{score} + B)$ |
| **Overnight Risk** | Gap & hold decisions | $P(\text{gap} > 2\%) \approx 0.20 + 0.30 \times frac{\Delta VIX}{20}$ |
| **Cross-Market Signals** | Risk transmission detection | $\text{Score} = 0.4 \Delta F + 0.25 \Delta USD + 0.2 \Delta Oil + 0.15 \Delta VIX$ |

---

## Further Reading

For full mathematical derivations, regime tables, and worked examples, see the accompanying **TRADING_GUIDE.md** (2,500+ lines):

- **Part I**: Foundation (Black-Scholes, Greeks, IV)
- **Part II**: Market microstructure (Dealer gamma, order flow)
- **Part III**: Regimes (Gamma/Vol/Macro/Global risk states)
- **Part IV**: Engine scoring (Strength, confirmation, confidence)
- **Part V**: Prediction (Rules, ML, calibration, policy, flow)
- **Part VII**: Advanced Topics (Cross-market, overnight risk, event-driven)

---

## Contact & Support

For questions or feedback on the trading system, refer to the weekly trading guide updates and email the research team.

**Last Updated**: April 2, 2026  
**Notebook Version**: 1.0

## Section 6: Options Flow Analysis & Dealer Positioning

**Flow Definition**: Volume × Price × |Delta| (delta-weighted notional flow)

**High call flow** (relative to puts) = Buyers are bullish → Upside bias
**High put flow** = Buyers are bearish → Downside bias

**Engine uses ATM slice only** (strike_window ±4 steps from current ATM) to avoid skew distortion.

**Thresholds**:
- **BULLISH_FLOW**: Call/Put ratio ≥ 1.2 → Take upside bias
- **BEARISH_FLOW**: Call/Put ratio ≤ 0.83 → Take downside bias
- **NEUTRAL_FLOW**: Between thresholds → No directional tilt

Below: Interactive flow calculator with volume and delta inputs.

## Section 5: Cross-Market Signals & Risk Transmission

**Key insight**: Indian equity options don't move in isolation. Risk transmits via:

1. **USD/INR (Currency Risk)**: Strong USD = INR weakness = corporates' USD debt → pressure
2. **Crude Oil (Inflation Signal)**: Oil crash = deflation fears = risk-off
3. **S&P 500 Futures (Overnight Spillover)**: S&P down → NIFTY opens down
4. **VIX (Volatility Contagion)**: VIX spike = panic spreading globally

**Quantitative Model** (Chapter 21):
$$\text{Cross-Market Score} = w_{\text{futures}} \Delta F + w_{\text{usd}} \Delta USD + w_{\text{oil}} \Delta Oil + w_{\text{vix}} \Delta VIX$$

Below: Interactive cross-market scorer with real thresholds.

## Section 4: Overnight Risk & Gap Mechanics

**Gap Definition**: Price difference when market reopens after extended closure.

**Overnight gaps** transfer from US markets (overnight when India closed):
- S&P down 2-3% → NIFTY opens 1.5-2.5% lower (100-200 pts)
- S&P down >3% → NIFTY opens 2.5-4% lower (200-400 pts)
- VIX spike >30 → Volatility contagion + gap

**Engine decision**: Hold overnight if RISK_OFF and NEGATIVE_GAMMA flags are false. Otherwise, exit before 15:15 IST.

Below: Gap probability calculator and overnight decision tree.

## Section 3: Probability Calibration (Platt Scaling)

**Problem**: Raw model scores (0-100) are NOT probabilities. They're just relative rankings.

**Solution**: Platt scaling transforms scores → calibrated probabilities via:

$$P(Y=1|\text{score}) = \frac{1}{1 + e^{-(A \cdot \text{score} + B)}}$$

**Training**: Gradient descent minimizes log-loss on validation set (2000 iterations).

Below: Interactive calibration curve with sliders for A, B parameters.

## Section 2: Regime Classification Engine

**Three orthogonal regime dimensions:**

1. **Gamma Regime**: Dealer inventory positioning (POSITIVE, NEGATIVE, NEUTRAL)
2. **Volatility Regime**: Vol term structure state (EXPANSION, COMPRESSION, NEUTRAL)
3. **Global Risk State**: Macro risk-off signals (HIGH_RISK, NORMAL, LOW_RISK)

**Thresholds** are extracted from the live engine codebase. Below: Interactive classifier with regime multipliers.

## Section 1: Greeks Calculator (Interactive)

**Theory**: Black-Scholes Greeks measure option price sensitivity to market parameters.

- **Delta (Δ)**: Price change per 1 unit move in spot
- **Gamma (Γ)**: Delta change per 1 unit move in spot (convexity)
- **Vega (ν)**: Price change per 1% move in volatility
- **Theta (Θ)**: Daily decay of option premium
- **Rho (ρ)**: Price change per 1% move in interest rate

Below: Interactive calculator with widgets for spot, strike, vol, and time.

# Options Quant Engine: Interactive Trading Guide

## A Hands-On Companion to the Comprehensive Trading Manual

**Version 1.0** — April 2, 2026

---

This notebook provides **interactive, executable examples** for the Options Quant Engine trading framework. Traders can:

1. **Calculate Greeks** (Delta, Gamma, Vega, Theta) with live spot/IV inputs
2. **Classify market regimes** (gamma/volatility/macro states)
3. **Calibrate probabilities** using Platt scaling
4. **Model overnight risk** and gap mechanics
5. **Analyze cross-market signals** (USD/INR, crude oil, VIX)
6. **Simulate signal generation** with real snapshot data
7. **Backtest strategies** with transaction costs

Use **ipywidgets** to control thresholds and parameters in real-time.